# ARBench: ToC-structured JSON chunks (Google Colab, L4 GPU)

This is the **JSON variant** of `ARBench_Colab_L4.ipynb`. The models, benchmark tasks and reports are the same.
What changes is how the annual report is chunked and what the SLMs read:

| | original notebook (`packed`) | this notebook (`toc_json`) |
|---|---|---|
| chunk boundaries | paragraphs/tables packed into ~700-token windows | **cut along the report's table of contents**; a chunk never crosses a section |
| ToC source | - | PDF bookmarks → printed *Contents* page (page offset learned) → Markdown headings |
| chunk format | flat Markdown text | **JSON**: `section.path`, `subsection`, `pages`, `part`, typed `content` blocks |
| what the SLMs read | text excerpts | the JSON chunks (generation, verification, evaluation, judging) |

```json
{
 "document": "John Keells Holdings PLC Annual Report 2025/26",
 "section": {"title": "OUTLOOK AND RISKS", "path": ["GROUP CONSOLIDATED REVIEW", "OUTLOOK AND RISKS"], "level": 2},
 "subsection": "Conflict in the Middle East",
 "pages": {"start": 139, "end": 140},
 "part": {"index": 2, "of": 26},
 "content": [
  {"type": "paragraph", "text": "..."},
  {"type": "list", "items": ["...", "..."]},
  {"type": "table", "columns": ["", "2026", "2025"], "rows": [["Revenue", "...", "..."]]}
 ]
}
```

**Before you start:** `Runtime → Change runtime type → L4 GPU`.

Steps: 1) get the project · 2) install Ollama · 3) models + annual report · 4) **inspect the ToC and JSON chunks** ·
5) run the benchmark on JSON chunks · 6) results · 7) optional controlled A/B (JSON vs. flat-text context, same questions) · 8) download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Get the project

Same as the original notebook. **Option A:** drag `ARBench.zip` (or `ARBench.zip.part00`, `.part01`, …) into Colab's
**Files** pane. **Option B:** set `USE_DRIVE = True` and point `PROJECT_DIR` at the project folder on Google Drive.
If you run this notebook locally in Jupyter (not Colab), this cell just uses the project folder.

The zip must contain this version of the code (with `arbench/pipeline/json_chunk.py`). Re-zip the project if your zip is older.

In [ ]:
import os, sys, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False                                  # True -> use the project folder on Google Drive
PROJECT_DIR = "/content/drive/Othercomputers/My laptop/Lectures/Year 2/DSGP/Benchmarking SLMs/Local Benchmarking with Evals"

if not IN_COLAB:                                   # local Jupyter: the notebook lives in <project>/notebooks
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    parts = sorted(glob.glob("/content/ARBench.zip.part*"))
    if parts and not os.path.exists("/content/ARBench.zip"):       # browser uploads are size-limited -> split zip
        with open("/content/ARBench.zip", "wb") as out:
            for part in parts:
                out.write(open(part, "rb").read())
        print("Joined", len(parts), "parts")
    zips = sorted(glob.glob("/content/ARBench*.zip") + glob.glob("/content/*.zip"))
    if not zips:
        from google.colab import files
        print("Upload ARBench.zip (the zipped project folder):")
        zips = ["/content/" + next(iter(files.upload()))]
    print("Extracting", zips[0])
    zipfile.ZipFile(zips[0]).extractall("/content/ARBench_src")
    hits = glob.glob("/content/ARBench_src/**/run_benchmark.py", recursive=True)
    assert hits, "run_benchmark.py not found inside the zip"
    PROJECT_DIR = os.path.dirname(hits[0])

assert os.path.exists(os.path.join(PROJECT_DIR, "run_benchmark.py")), f"run_benchmark.py not found in {PROJECT_DIR}"
assert os.path.exists(os.path.join(PROJECT_DIR, "arbench", "pipeline", "json_chunk.py")), \
    "This project copy is too old: it has no arbench/pipeline/json_chunk.py. Re-zip the current project."
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("Project:", os.getcwd())

from IPython.display import Markdown, display
def show_table(rows):   # list of dicts -> Markdown table (no pandas needed)
    if not rows:
        return print("(no rows)")
    cols = list(dict.fromkeys(k for r in rows for k in r))
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join("" if r.get(c) is None else str(r[c]) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## 2 · Install Ollama + Python dependencies and start the server (GPU)

In [ ]:
if IN_COLAB:
    !apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install -r requirements.txt

In [ ]:
import subprocess, time, requests, os
def ollama_up():
    try:
        return requests.get("http://127.0.0.1:11434/api/version", timeout=2).json()
    except Exception:
        return None

if not ollama_up():
    env = dict(os.environ,
               OLLAMA_NUM_PARALLEL="4",        # parallel requests per model (matches --profile colab)
               OLLAMA_MAX_LOADED_MODELS="3",   # all three SLMs fit in 24 GB VRAM
               OLLAMA_KEEP_ALIVE="30m",
               OLLAMA_FLASH_ATTENTION="1")
    log = "/content/ollama.log" if IN_COLAB else "ollama.log"
    server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("Ollama", ollama_up())

## 3 · Choose the models and add the annual report

In [ ]:
MODELS = ["granite4.1:3b", "gemma4:e2b-it-qat", "qwen3:4b-instruct"]
# Larger variants that fit an L4 comfortably:
# MODELS = ["granite4.1:8b", "gemma4:e4b-it-qat", "qwen3:8b"]
import subprocess
for m in MODELS:
    for attempt in range(5):                       # retry flaky downloads
        if subprocess.run(["ollama", "pull", m]).returncode == 0:
            break
    else:
        raise RuntimeError(f"could not pull {m}")
!ollama list

In [ ]:
import os
REPORT_DIR = "data/annual_report"
os.makedirs(REPORT_DIR, exist_ok=True)
existing = [f for f in os.listdir(REPORT_DIR) if f.lower().endswith((".pdf", ".docx", ".html", ".htm", ".md", ".txt")) and f.lower() != "readme.md"]
if not existing and IN_COLAB:
    from google.colab import files
    print("Upload the annual report (PDF/DOCX/HTML/MD/TXT):")
    for name, data in files.upload().items():
        open(os.path.join(REPORT_DIR, name), "wb").write(data)
print("Reports:", [f for f in os.listdir(REPORT_DIR) if f.lower() != "readme.md"])

## 4 · Inspect the table of contents and the JSON chunks (no LLM calls)

This runs the same ingestion and ToC-aware JSON chunking that the benchmark uses, so you can check the structure
before spending GPU time. Text extraction takes a few minutes for a ~600-page PDF and is cached, so the benchmark run reuses it.

In [ ]:
import json, logging, collections
from pathlib import Path
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
from arbench.config import load_config, resolve_path, PROJECT_ROOT
from arbench.pipeline.ingest import run_ingestion
from arbench.pipeline.json_chunk import run_json_chunking

cfg = load_config(profile="colab", overrides={"chunking": {"strategy": "toc_json"}})
docs = run_ingestion(resolve_path(cfg["input_dir"]), cfg["ingestion"], PROJECT_ROOT, Path("outputs/json_preview/01_extracted"),
                     resolve_path(cfg["cache_path"]).parent / "extracted")
chunks, tocs = run_json_chunking(docs, cfg["chunking"])

for toc in tocs:
    print(f"\n{toc['document']}: ToC source = {toc['toc_source']} · {toc['n_entries']} entries · "
          f"{sum(e['located'] for e in toc['entries'])} pinned to a heading line · {toc['n_sections']} sections · "
          f"{toc['n_kept']}/{toc['n_chunks']} JSON chunks kept")
    for e in toc["entries"]:
        print(f"  {'  ' * (len(e['path']) - 1)}{e['title'][:70]:<70}  p.{e['page']:<4} {'' if e['located'] else '(top of page)'}")

In [ ]:
import numpy as np
per_section = collections.Counter(c.section for c in chunks)
toks = np.array([c.n_tokens for c in chunks])
blocks = collections.Counter(b["type"] for c in chunks for b in c.payload["content"])
print(f"{len(chunks)} JSON chunks · tokens per chunk: median {np.median(toks):.0f}, p95 {np.percentile(toks, 95):.0f}, max {toks.max()}")
print("content blocks:", dict(blocks))
show_table([{"section path": s, "chunks": n} for s, n in per_section.most_common(25)])

### A JSON chunk and the exact prompts the SLMs receive

Change `PAGE` to look at another part of the report.

In [ ]:
from arbench.pipeline.generate import _single_request
from arbench.evals.tasks import _item, build_messages

PAGE = 139
c = next((c for c in chunks if PAGE in c.pages), chunks[len(chunks) // 2])
print(c.prompt_text)

gen = _single_request("single_hop", c, docs[0].title, "<document summary>", 3, 0.4)
print("\n" + "=" * 100 + "\nQUESTION GENERATION - system prompt (end):\n", gen.messages[0]["content"][-900:])
fake_q = {"qid": "demo", "kind": "single_hop", "question": "<question>", "answer": "<gold>", "question_type": "factual",
          "difficulty": 3, "generator": MODELS[0], "doc_id": c.doc_id, "pages": c.pages, "chunk_ids": [c.chunk_id]}
ev = build_messages(_item("grounded_qa", fake_q, [c], [c]))
print("\n" + "=" * 100 + "\nEVALUATION - system prompt:\n", ev[0]["content"])
print("\nEVALUATION - user prompt (start):\n", ev[1]["content"][:700])
Path("outputs/json_preview").mkdir(parents=True, exist_ok=True)
json.dump([{"chunk_id": c.chunk_id, **c.payload} for c in chunks], open("outputs/json_preview/chunks.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump(tocs, open("outputs/json_preview/toc.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\nSaved outputs/json_preview/chunks.json and toc.json")

## 5 · Run the whole benchmark on JSON chunks (one command)

`--chunking toc_json` switches the pipeline to the ToC-structured JSON chunks; everything else is identical to the original
notebook (`--profile colab` = full coverage with 4 parallel requests; use `--profile quick` for a fast first look).
All LLM calls are cached, so re-running the cell resumes an interrupted run.

In [ ]:
RUN = "colab_json_run"
!python run_benchmark.py --profile colab --chunking toc_json --run-name {RUN} --models {" ".join(MODELS)}

## 6 · Results

In [ ]:
from IPython.display import Markdown, display, Image
run = f"outputs/{RUN}"
display(Markdown(open(f"{run}/reports/comparison_report.md", encoding="utf-8").read().replace("](charts/", f"]({run}/reports/charts/")))

In [ ]:
for c in ["leaderboard", "tasks", "quality_vs_speed", "hallucination", "question_types", "difficulty"]:
    p = f"outputs/{RUN}/reports/charts/{c}.png"
    if os.path.exists(p): display(Image(p))

## 7 · Optional: controlled A/B. Does reading JSON help the SLMs?

Comparing this run with the original `packed` run mixes two effects: different chunks *and* different generated questions.
This step isolates the **format**. It copies the JSON benchmark and re-renders every model-facing context as flat text
**from the same chunks**. Questions, gold answers, retrieved chunks and the judge's evidence stay identical. Only the
excerpt format changes, so the per-item differences are paired.

In [ ]:
from arbench.context_ab import make_text_context_benchmark
AB_BENCH, AB_RUN = "outputs/colab_json_textctx_bench", "colab_json_textctx"
make_text_context_benchmark(f"outputs/{RUN}", AB_BENCH)
!python run_benchmark.py --profile colab --reuse-benchmark {AB_BENCH} --run-name {AB_RUN} --models {" ".join(MODELS)}

In [ ]:
import glob, numpy as np
from arbench.evals.scoring import primary_score
from arbench.utils import read_jsonl

ARMS = {"JSON context (toc_json)": f"outputs/{RUN}",
        "flat-text context, same chunks (A/B)": f"outputs/{AB_RUN}",
        "packed text chunks (original notebook)": "outputs/colab_run"}    # shown if that run is present

def load_preds(run_dir):
    out = {}
    for f in glob.glob(f"{run_dir}/predictions/*.jsonl"):
        rows = read_jsonl(f)
        if rows:
            out[rows[0]["model"]] = {r["item_id"]: r for r in rows}
    return out

rows = []
for arm, d in ARMS.items():
    f = f"{d}/results/results.json"
    if not os.path.exists(f):
        continue
    r = json.load(open(f, encoding="utf-8"))
    preds = load_preds(d)
    for m, s in r["scores"].items():
        row = {"setup": arm, "model": r["models"][m]["label"], "ARBench": round(100 * s["composite"], 1)}
        row.update({t: round(100 * ts["primary"]["mean"], 1) for t, ts in s["tasks"].items()})
        pt = [p["stats"]["prompt_tokens"] for p in preds.get(m, {}).values() if p["stats"].get("prompt_tokens")]
        row["mean prompt tokens"] = round(float(np.mean(pt))) if pt else None
        row["gen tok/s"] = round(s["efficiency"]["gen_tokens_per_s"], 1)
        rows.append(row)
show_table(sorted(rows, key=lambda r: (r["model"], r["setup"])))

In [ ]:
# paired comparison on identical items: JSON minus flat-text, with a bootstrap 95% CI
judged = json.load(open(f"outputs/{RUN}/config_used.json", encoding="utf-8"))["judge"]["enabled"]
pj, pt = load_preds(f"outputs/{RUN}"), load_preds(f"outputs/{AB_RUN}")
rng = np.random.default_rng(0)
out = []
for m in pj:
    by_task = collections.defaultdict(list)
    for iid, a in pj[m].items():
        b = pt.get(m, {}).get(iid)
        if b is None:
            continue
        sa, sb = primary_score(a["task"], a["scores"], judged), primary_score(b["task"], b["scores"], judged)
        if sa is not None and sb is not None:
            by_task[a["task"]].append(sa - sb)
    for task, diffs in sorted(by_task.items()):
        d = np.array(diffs)
        boots = rng.choice(d, (2000, len(d))).mean(1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({"model": m, "task": task, "n": len(d), "JSON - text (pts)": round(100 * d.mean(), 1),
                    "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]",
                    "JSON better / worse / tie": f"{(d > 0).sum()} / {(d < 0).sum()} / {(d == 0).sum()}"})
show_table(out)
print("A CI that excludes 0 means the context format made a real difference for that model and task.")

## 8 · Download the results

In [ ]:
import shutil
targets = [d for d in [f"outputs/{RUN}", f"outputs/{AB_RUN}" if "AB_RUN" in dir() else None, "outputs/json_preview"] if d and os.path.exists(d)]
os.makedirs("/content/arbench_json_results" if IN_COLAB else "arbench_json_results", exist_ok=True)
dest = "/content/arbench_json_results" if IN_COLAB else "arbench_json_results"
for t in targets:
    shutil.copytree(t, os.path.join(dest, os.path.basename(t)), dirs_exist_ok=True)
archive = shutil.make_archive(dest, "zip", dest)
print("Archive:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)   # also saved inside PROJECT_DIR/outputs if you used Drive